# Visit with Us — Wellness Tourism Package Prediction

**MLOps project notebook | Google Colab compatible**

## Business objective
Predict whether a customer purchases the Wellness Tourism Package (`ProdTaken`: 0 = No, 1 = Yes). The workflow validates the dataset, cleans and prepares data, compares classifiers, tunes a model, evaluates it, saves deployment artifacts, and creates a Streamlit application and GitHub Actions workflow.

**Submission reminder:** The supplied rubric asks for an executed notebook exported to **HTML**. Keep all outputs visible, then use **Runtime → Run all**, download the `.ipynb`, and export/convert it to `.html`.

**Observation:** This notebook follows the project rubric and uses the supplied `tourism.csv`. The notebook is a working development artifact; public GitHub and deployment URLs must be added after you create those accounts/resources.

## 1. Install and import dependencies
The project uses pandas and scikit-learn for data processing/model development, joblib for model persistence, and Streamlit for the frontend.

In [ ]:
# Google Colab usually has pandas and scikit-learn preinstalled.
# Uncomment the next line only if an import below reports a missing package.
# %pip -q install pandas numpy scikit-learn joblib streamlit
import pandas, numpy, sklearn, joblib
print("Core ML dependencies are available.")

**Observation:** Installing the dependencies makes the notebook environment consistent with the deployment requirements. The same packages are listed in `requirements.txt` for GitHub Actions and the app.

## 2. Upload / locate the dataset and project script
Upload `tourism.csv` if it is not already available in the notebook session. The Python pipeline file is included in the companion project ZIP; this notebook also embeds it below so the workflow can be run from the notebook alone.

In [ ]:
from pathlib import Path
import os, base64, subprocess, sys, json

# The dataset is already mounted in this prepared copy. In Colab, upload tourism.csv if needed.
data_candidates = [Path('/content/tourism.csv'), Path('/mnt/data/tourism.csv'), Path('tourism.csv')]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    from google.colab import files
    uploaded = files.upload()
    data_path = Path(next(iter(uploaded)))
print('Dataset selected:', data_path)
print('File size (KB):', round(data_path.stat().st_size / 1024, 1))

**Observation:** The notebook selects the uploaded CSV without requiring a fixed local path. In Colab, use the upload prompt if the dataset is not already present.

In [ ]:
# Write the companion, reusable Python pipeline into the current notebook workspace.
from pathlib import Path
script_b64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwojIFZJU0lUIFdJVEggVVMgLSBXRUxMTkVTUyBUT1VSSVNNIFBBQ0tBR0UgUFJFRElDVElPTgojIFJ1biBpbiBDb2xhYjogIXB5dGhvbiB0b3VyaXNtX3BhY2thZ2VfbWxvcHMucHkgLS1kYXRhIC9jb250ZW50L3RvdXJpc20uY3N2CiMgT3V0cHV0czogdHJhaW4vdGVzdCBzcGxpdHMsIHRyYWluZWQgbW9kZWwsIHJlcG9ydHMsIFN0cmVhbWxpdCBhcHAgYW5kIEdpdEh1YiBBY3Rpb25zIHdvcmtmbG93LgoKaW1wb3J0IGFyZ3BhcnNlLCBqc29uLCBzaHV0aWwsIHdhcm5pbmdzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAp3YXJuaW5ncy5maWx0ZXJ3YXJuaW5ncygnaWdub3JlJykKRVhQRUNURUQgPSBbJ0N1c3RvbWVySUQnLCdQcm9kVGFrZW4nLCdBZ2UnLCdUeXBlb2ZDb250YWN0JywnQ2l0eVRpZXInLCdEdXJhdGlvbk9mUGl0Y2gnLAonT2NjdXBhdGlvbicsJ0dlbmRlcicsJ051bWJlck9mUGVyc29uVmlzaXRpbmcnLCdOdW1iZXJPZkZvbGxvd3VwcycsJ1Byb2R1Y3RQaXRjaGVkJywKJ1ByZWZlcnJlZFByb3BlcnR5U3RhcicsJ01hcml0YWxTdGF0dXMnLCdOdW1iZXJPZlRyaXBzJywnUGFzc3BvcnQnLCdQaXRjaFNhdGlzZmFjdGlvblNjb3JlJywKJ093bkNhcicsJ051bWJlck9mQ2hpbGRyZW5WaXNpdGluZycsJ0Rlc2lnbmF0aW9uJywnTW9udGhseUluY29tZSddClRBUkdFVCwgU0VFRCA9ICdQcm9kVGFrZW4nLCA0MgoKZGVmIG9icyh0ZXh0KToKICAgIHByaW50KCdcbicgKyAnPScqMTggKyAnIE9CU0VSVkFUSU9OICcgKyAnPScqMTgpCiAgICBwcmludCh0ZXh0KQogICAgcHJpbnQoJz0nKjUwICsgJ1xuJykKCmRlZiBtYWluKCk6CiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkLCBudW1weSBhcyBucCwgam9ibGliCiAgICBmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0LCBHcmlkU2VhcmNoQ1YsIFN0cmF0aWZpZWRLRm9sZAogICAgZnJvbSBza2xlYXJuLmNvbXBvc2UgaW1wb3J0IENvbHVtblRyYW5zZm9ybWVyCiAgICBmcm9tIHNrbGVhcm4ucGlwZWxpbmUgaW1wb3J0IFBpcGVsaW5lCiAgICBmcm9tIHNrbGVhcm4uaW1wdXRlIGltcG9ydCBTaW1wbGVJbXB1dGVyCiAgICBmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgT25lSG90RW5jb2RlciwgU3RhbmRhcmRTY2FsZXIKICAgIGZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExvZ2lzdGljUmVncmVzc2lvbgogICAgZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBSYW5kb21Gb3Jlc3RDbGFzc2lmaWVyLCBFeHRyYVRyZWVzQ2xhc3NpZmllcgogICAgZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGFjY3VyYWN5X3Njb3JlLCBwcmVjaXNpb25fc2NvcmUsIHJlY2FsbF9zY29yZSwgZjFfc2NvcmUsIHJvY19hdWNfc2NvcmUsIGNvbmZ1c2lvbl9tYXRyaXgsIGNsYXNzaWZpY2F0aW9uX3JlcG9ydAoKICAgIHBhcnNlcj1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWRhdGEnLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXJvb3QnLCBkZWZhdWx0PSd0b3VyaXNtX21sb3BzX3Byb2plY3QnKQogICAgYXJncz1wYXJzZXIucGFyc2VfYXJncygpCiAgICByb290PVBhdGgoYXJncy5yb290KS5yZXNvbHZlKCkKICAgIGZvciBmb2xkZXIgaW4gWydkYXRhJywnYXJ0aWZhY3RzJywnZGVwbG95bWVudCcsJ3JlcG9ydHMnLCcuZ2l0aHViL3dvcmtmbG93cyddOgogICAgICAgIChyb290L2ZvbGRlcikubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICB0aGlzX3NjcmlwdD1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkKICAgIGlmIHRoaXNfc2NyaXB0ICE9IChyb290Lyd0b3VyaXNtX3BhY2thZ2VfbWxvcHMucHknKS5yZXNvbHZlKCk6CiAgICAgICAgc2h1dGlsLmNvcHkyKHRoaXNfc2NyaXB0LHJvb3QvJ3RvdXJpc21fcGFja2FnZV9tbG9wcy5weScpCgogICAgY2FuZGlkYXRlcz1bUGF0aChhcmdzLmRhdGEpXSBpZiBhcmdzLmRhdGEgZWxzZSBbXQogICAgY2FuZGlkYXRlcyArPSBbUGF0aCgndG91cmlzbS5jc3YnKSxQYXRoKCcvY29udGVudC90b3VyaXNtLmNzdicpLFBhdGgoJy9tbnQvZGF0YS90b3VyaXNtLmNzdicpXQogICAgc291cmNlPW5leHQoKHAucmVzb2x2ZSgpIGZvciBwIGluIGNhbmRpZGF0ZXMgaWYgcC5leGlzdHMoKSksTm9uZSkKICAgIGlmIHNvdXJjZSBpcyBOb25lOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IGZpbGVzCiAgICAgICAgICAgIHByaW50KCdQbGVhc2UgdXBsb2FkIHRvdXJpc20uY3N2JykKICAgICAgICAgICAgdXBsb2FkZWQ9ZmlsZXMudXBsb2FkKCkKICAgICAgICAgICAgc291cmNlPVBhdGgobmV4dChpdGVyKHVwbG9hZGVkKSkpLnJlc29sdmUoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKCdQbGFjZSB0b3VyaXNtLmNzdiBiZXNpZGUgdGhpcyBzY3JpcHQgb3IgcGFzcyAtLWRhdGEgL3BhdGgvdG91cmlzbS5jc3YnKQogICAgY3N2X3BhdGg9cm9vdC8nZGF0YS90b3VyaXNtLmNzdicKICAgIGlmIHNvdXJjZSAhPSBjc3ZfcGF0aC5yZXNvbHZlKCk6IHNodXRpbC5jb3B5Mihzb3VyY2UsY3N2X3BhdGgpCiAgICBkZj1wZC5yZWFkX2Nzdihjc3ZfcGF0aCkKCiAgICBtaXNzaW5nPXNvcnRlZChzZXQoRVhQRUNURUQpLXNldChkZi5jb2x1bW5zKSkKICAgIGlmIG1pc3Npbmc6IHJhaXNlIFZhbHVlRXJyb3IoZidNaXNzaW5nIGV4cGVjdGVkIGNvbHVtbnM6IHttaXNzaW5nfScpCiAgICBpZiBkZltUQVJHRVRdLm51bmlxdWUoKTwyOiByYWlzZSBWYWx1ZUVycm9yKCdQcm9kVGFrZW4gbXVzdCBjb250YWluIGJvdGggMCBhbmQgMS4nKQogICAgcHJpbnQoJ0RBVEEgUkVHSVNUUkFUSU9OJykKICAgIHByaW50KCdTaGFwZTonLGRmLnNoYXBlLCdcblRhcmdldCBjb3VudHM6XG4nLGRmW1RBUkdFVF0udmFsdWVfY291bnRzKCkuc29ydF9pbmRleCgpKQogICAgb2JzKGYnRGF0YXNldCByZWdpc3RlcmVkIHdpdGgge2xlbihkZik6LH0gcm93cyBhbmQge2RmLnNoYXBlWzFdfSBjb2x1bW5zLiBBbGwgZXhwZWN0ZWQgZmllbGRzIGFyZSBwcmVzZW50LiBSZXZpZXcgdGFyZ2V0IGNvdW50cyBhYm92ZSB0byB1bmRlcnN0YW5kIGNsYXNzIGJhbGFuY2UuJykKCiAgICAjIENsZWFuIGR1cGxpY2F0ZXMsIHJlbW92ZSByb3dzIHdpdGhvdXQgdGFyZ2V0IGFuZCBkcm9wIHJlY29yZCBpZGVudGlmaWVycy4KICAgIGJlZm9yZT1sZW4oZGYpCiAgICBkZj1kZi5kcm9wX2R1cGxpY2F0ZXMoKS5kcm9wbmEoc3Vic2V0PVtUQVJHRVRdKS5jb3B5KCkKICAgIGlkX2NvbHM9W2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjLmxvd2VyKCkuc3RhcnRzd2l0aCgndW5uYW1lZCcpIG9yIGM9PSdDdXN0b21lcklEJ10KICAgIGRmPWRmLmRyb3AoY29sdW1ucz1pZF9jb2xzLGVycm9ycz0naWdub3JlJykKICAgIGRmW1RBUkdFVF09cGQudG9fbnVtZXJpYyhkZltUQVJHRVRdLGVycm9ycz0nY29lcmNlJykKICAgIGRmPWRmLmRyb3BuYShzdWJzZXQ9W1RBUkdFVF0pCiAgICBkZltUQVJHRVRdPWRmW1RBUkdFVF0uYXN0eXBlKGludCkKICAgIGlmIG5vdCBzZXQoZGZbVEFSR0VUXS51bmlxdWUoKSkuaXNzdWJzZXQoezAsMX0pOiByYWlzZSBWYWx1ZUVycm9yKCdQcm9kVGFrZW4gc2hvdWxkIGNvbnRhaW4gb25seSAwIGFuZCAxLicpCiAgICBYLHk9ZGYuZHJvcChjb2x1bW5zPVRBUkdFVCksZGZbVEFSR0VUXQogICAgWF90cmFpbixYX3Rlc3QseV90cmFpbix5X3Rlc3Q9dHJhaW5fdGVzdF9zcGxpdChYLHksdGVzdF9zaXplPS4yMCxyYW5kb21fc3RhdGU9U0VFRCxzdHJhdGlmeT15KQogICAgdHJhaW49WF90cmFpbi5jb3B5KCk7IHRyYWluW1RBUkdFVF09eV90cmFpbgogICAgdGVzdD1YX3Rlc3QuY29weSgpOyB0ZXN0W1RBUkdFVF09eV90ZXN0CiAgICB0cmFpbi50b19jc3Yocm9vdC8nYXJ0aWZhY3RzL3RyYWluLmNzdicsaW5kZXg9RmFsc2UpCiAgICB0ZXN0LnRvX2Nzdihyb290LydhcnRpZmFjdHMvdGVzdC5jc3YnLGluZGV4PUZhbHNlKQogICAgcHJpbnQoJ1BSRVBBUkFUSU9OOiBiZWZvcmU9JyxiZWZvcmUsJ2FmdGVyPScsbGVuKGRmKSwnZHJvcHBlZCBpZGVudGlmaWVycz0nLGlkX2NvbHMpCiAgICBwcmludCgnVHJhaW4gc2hhcGU6Jyx0cmFpbi5zaGFwZSwnVGVzdCBzaGFwZTonLHRlc3Quc2hhcGUpCiAgICBvYnMoZidBZnRlciByZW1vdmluZyBkdXBsaWNhdGVzIGFuZCByb3dzIHdpdGhvdXQgYSB0YXJnZXQsIHtsZW4oZGYpOix9IHJlY29yZHMgcmVtYWluLiBBbiA4MDoyMCBzdHJhdGlmaWVkIHNwbGl0IGNyZWF0ZWQge2xlbih0cmFpbik6LH0gdHJhaW5pbmcgYW5kIHtsZW4odGVzdCk6LH0gdGVzdCByb3dzLiBJZGVudGlmaWVyIGNvbHVtbnMge2lkX2NvbHN9IHdlcmUgcmVtb3ZlZC4gTWlzc2luZyBwcmVkaWN0b3IgdmFsdWVzIGFyZSBpbXB1dGVkIHdpdGhpbiB0aGUgbW9kZWwgcGlwZWxpbmUuJykKCiAgICBudW1lcmljPVhfdHJhaW4uc2VsZWN0X2R0eXBlcyhpbmNsdWRlPSdudW1iZXInKS5jb2x1bW5zLnRvbGlzdCgpCiAgICBjYXRlZ29yaWNhbD1YX3RyYWluLnNlbGVjdF9kdHlwZXMoZXhjbHVkZT0nbnVtYmVyJykuY29sdW1ucy50b2xpc3QoKQogICAgcHJlcD1Db2x1bW5UcmFuc2Zvcm1lcihbCiAgICAgICgnbnVtJyxQaXBlbGluZShbKCdpbXB1dGVyJyxTaW1wbGVJbXB1dGVyKHN0cmF0ZWd5PSdtZWRpYW4nKSksKCdzY2FsZScsU3RhbmRhcmRTY2FsZXIoKSldKSxudW1lcmljKSwKICAgICAgKCdjYXQnLFBpcGVsaW5lKFsoJ2ltcHV0ZXInLFNpbXBsZUltcHV0ZXIoc3RyYXRlZ3k9J21vc3RfZnJlcXVlbnQnKSksKCdvbmVob3QnLE9uZUhvdEVuY29kZXIoaGFuZGxlX3Vua25vd249J2lnbm9yZScpKV0pLGNhdGVnb3JpY2FsKQogICAgXSkKICAgIGNhbmRpZGF0ZXM9ewogICAgICAnTG9naXN0aWMgUmVncmVzc2lvbic6TG9naXN0aWNSZWdyZXNzaW9uKG1heF9pdGVyPTIwMDAsY2xhc3Nfd2VpZ2h0PSdiYWxhbmNlZCcscmFuZG9tX3N0YXRlPVNFRUQpLAogICAgICAnUmFuZG9tIEZvcmVzdCc6UmFuZG9tRm9yZXN0Q2xhc3NpZmllcihuX2VzdGltYXRvcnM9MjUwLGNsYXNzX3dlaWdodD0nYmFsYW5jZWQnLHJhbmRvbV9zdGF0ZT1TRUVELG5fam9icz0xKSwKICAgICAgJ0V4dHJhIFRyZWVzJzpFeHRyYVRyZWVzQ2xhc3NpZmllcihuX2VzdGltYXRvcnM9MjUwLGNsYXNzX3dlaWdodD0nYmFsYW5jZWQnLHJhbmRvbV9zdGF0ZT1TRUVELG5fam9icz0xKQogICAgfQogICAgcm93cz1bXQogICAgZm9yIG5hbWUsZXN0IGluIGNhbmRpZGF0ZXMuaXRlbXMoKToKICAgICAgICBwaXBlPVBpcGVsaW5lKFsoJ3ByZXByb2Nlc3MnLHByZXApLCgnbW9kZWwnLGVzdCldKQogICAgICAgIHBpcGUuZml0KFhfdHJhaW4seV90cmFpbikKICAgICAgICBwcmVkPXBpcGUucHJlZGljdChYX3Rlc3QpOyBwcm9iYT1waXBlLnByZWRpY3RfcHJvYmEoWF90ZXN0KVs6LDFdCiAgICAgICAgcm93cy5hcHBlbmQoeydtb2RlbCc6bmFtZSwnYWNjdXJhY3knOmFjY3VyYWN5X3Njb3JlKHlfdGVzdCxwcmVkKSwncHJlY2lzaW9uJzpwcmVjaXNpb25fc2NvcmUoeV90ZXN0LHByZWQsemVyb19kaXZpc2lvbj0wKSwncmVjYWxsJzpyZWNhbGxfc2NvcmUoeV90ZXN0LHByZWQsemVyb19kaXZpc2lvbj0wKSwnZjEnOmYxX3Njb3JlKHlfdGVzdCxwcmVkLHplcm9fZGl2aXNpb249MCksJ3JvY19hdWMnOnJvY19hdWNfc2NvcmUoeV90ZXN0LHByb2JhKX0pCiAgICBjb21wYXJpc29uPXBkLkRhdGFGcmFtZShyb3dzKS5zb3J0X3ZhbHVlcyhbJ3JvY19hdWMnLCdmMSddLGFzY2VuZGluZz1GYWxzZSkKICAgIGNvbXBhcmlzb24udG9fY3N2KHJvb3QvJ3JlcG9ydHMvbW9kZWxfY29tcGFyaXNvbi5jc3YnLGluZGV4PUZhbHNlKQogICAgcHJpbnQoJ0JBU0VMSU5FIENPTVBBUklTT05cbicsY29tcGFyaXNvbi50b19zdHJpbmcoaW5kZXg9RmFsc2UpKQogICAgYmVzdHJvdz1jb21wYXJpc29uLmlsb2NbMF0KICAgIG9icyhmIntiZXN0cm93Wydtb2RlbCddfSBhY2hpZXZlZCB0aGUgaGlnaGVzdCBiYXNlbGluZSBST0MtQVVDICh7YmVzdHJvd1sncm9jX2F1YyddOi4zZn0pIG9uIHRoZSBoZWxkLW91dCB0ZXN0IGRhdGEuIENvbXBhcmUgcHJlY2lzaW9uIGFuZCByZWNhbGwgYXMgd2VsbDogcHJlY2lzaW9uIHJlbGF0ZXMgdG8gZWZmaWNpZW50IG1hcmtldGluZyBvdXRyZWFjaCwgd2hpbGUgcmVjYWxsIHJlbGF0ZXMgdG8gaG93IG1hbnkgYWN0dWFsIHB1cmNoYXNlcnMgYXJlIGlkZW50aWZpZWQuIikKCiAgICAjIFR1bmUgUmFuZG9tIEZvcmVzdCB1c2luZyBzdHJhdGlmaWVkIGNyb3NzLXZhbGlkYXRpb24uCiAgICBwaXBlPVBpcGVsaW5lKFsoJ3ByZXByb2Nlc3MnLHByZXApLCgnbW9kZWwnLFJhbmRvbUZvcmVzdENsYXNzaWZpZXIoY2xhc3Nfd2VpZ2h0PSdiYWxhbmNlZCcscmFuZG9tX3N0YXRlPVNFRUQsbl9qb2JzPTEpKV0pCiAgICBncmlkPXsnbW9kZWxfX25fZXN0aW1hdG9ycyc6WzE1MCwyNTBdLCdtb2RlbF9fbWF4X2RlcHRoJzpbTm9uZSwxMl0sJ21vZGVsX19taW5fc2FtcGxlc19zcGxpdCc6WzIsNV0sJ21vZGVsX19taW5fc2FtcGxlc19sZWFmJzpbMSwyXX0KICAgIHNlYXJjaD1HcmlkU2VhcmNoQ1YocGlwZSxncmlkLHNjb3Jpbmc9J3JvY19hdWMnLGN2PVN0cmF0aWZpZWRLRm9sZChuX3NwbGl0cz0zLHNodWZmbGU9VHJ1ZSxyYW5kb21fc3RhdGU9U0VFRCksbl9qb2JzPTEsdmVyYm9zZT0xKQogICAgc2VhcmNoLmZpdChYX3RyYWluLHlfdHJhaW4pCiAgICBtb2RlbD1zZWFyY2guYmVzdF9lc3RpbWF0b3JfCiAgICBwcmVkPW1vZGVsLnByZWRpY3QoWF90ZXN0KTsgcHJvYmE9bW9kZWwucHJlZGljdF9wcm9iYShYX3Rlc3QpWzosMV0KICAgIG1ldHJpY3M9eydiZXN0X21vZGVsJzonVHVuZWQgUmFuZG9tIEZvcmVzdCcsJ2Jlc3RfY3Zfcm9jX2F1Yyc6ZmxvYXQoc2VhcmNoLmJlc3Rfc2NvcmVfKSwndGVzdF9hY2N1cmFjeSc6ZmxvYXQoYWNjdXJhY3lfc2NvcmUoeV90ZXN0LHByZWQpKSwndGVzdF9wcmVjaXNpb24nOmZsb2F0KHByZWNpc2lvbl9zY29yZSh5X3Rlc3QscHJlZCx6ZXJvX2RpdmlzaW9uPTApKSwndGVzdF9yZWNhbGwnOmZsb2F0KHJlY2FsbF9zY29yZSh5X3Rlc3QscHJlZCx6ZXJvX2RpdmlzaW9uPTApKSwndGVzdF9mMSc6ZmxvYXQoZjFfc2NvcmUoeV90ZXN0LHByZWQsemVyb19kaXZpc2lvbj0wKSksJ3Rlc3Rfcm9jX2F1Yyc6ZmxvYXQocm9jX2F1Y19zY29yZSh5X3Rlc3QscHJvYmEpKSwnYmVzdF9wYXJhbXMnOnNlYXJjaC5iZXN0X3BhcmFtc18sJ2NvbmZ1c2lvbl9tYXRyaXgnOmNvbmZ1c2lvbl9tYXRyaXgoeV90ZXN0LHByZWQpLnRvbGlzdCgpLCdjbGFzc2lmaWNhdGlvbl9yZXBvcnQnOmNsYXNzaWZpY2F0aW9uX3JlcG9ydCh5X3Rlc3QscHJlZCxvdXRwdXRfZGljdD1UcnVlLHplcm9fZGl2aXNpb249MCl9CiAgICBqb2JsaWIuZHVtcChtb2RlbCxyb290LydkZXBsb3ltZW50L21vZGVsLmpvYmxpYicpCiAgICAocm9vdC8ncmVwb3J0cy9tZXRyaWNzLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWV0cmljcyxpbmRlbnQ9MikpCiAgICBwcmludCgnQkVTVCBQQVJBTUVURVJTOicsc2VhcmNoLmJlc3RfcGFyYW1zXykKICAgIGZvciBrIGluIFsndGVzdF9hY2N1cmFjeScsJ3Rlc3RfcHJlY2lzaW9uJywndGVzdF9yZWNhbGwnLCd0ZXN0X2YxJywndGVzdF9yb2NfYXVjJ106IHByaW50KGsscm91bmQobWV0cmljc1trXSw0KSkKICAgIHByaW50KCdDb25mdXNpb24gbWF0cml4IFtbVE4sRlBdLFtGTixUUF1dOlxuJyxucC5hcnJheShtZXRyaWNzWydjb25mdXNpb25fbWF0cml4J10pKQogICAgb2JzKGYiVGhlIHR1bmVkIFJhbmRvbSBGb3Jlc3QgdGVzdCBST0MtQVVDIGlzIHttZXRyaWNzWyd0ZXN0X3JvY19hdWMnXTouM2Z9OyBGMSBpcyB7bWV0cmljc1sndGVzdF9mMSddOi4zZn0sIHByZWNpc2lvbiBpcyB7bWV0cmljc1sndGVzdF9wcmVjaXNpb24nXTouM2Z9LCBhbmQgcmVjYWxsIGlzIHttZXRyaWNzWyd0ZXN0X3JlY2FsbCddOi4zZn0uIFVzZSB0aGUgY29uZnVzaW9uIG1hdHJpeCB0byBkaXNjdXNzIGZhbHNlIHBvc2l0aXZlcyBhbmQgbWlzc2VkIGJ1eWVycy4gVGhlc2UgbWV0cmljcyBkZXNjcmliZSB0aGlzIHRlc3Qgc3BsaXQgYW5kIHNob3VsZCBiZSBjaGVja2VkIGFnYWluIG9uIG5ld2VyIGRhdGEuIikKCiAgICAjIERlcGxveW1lbnQgYXBwbGljYXRpb24gKG1vZGVsIHBpcGVsaW5lIGluY2x1ZGVzIGFsbCBwcmVwcm9jZXNzaW5nKS4KICAgIGFwcD0nJydpbXBvcnQgc3RyZWFtbGl0IGFzIHN0CmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IGpvYmxpYgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKc3Quc2V0X3BhZ2VfY29uZmlnKHBhZ2VfdGl0bGU9IldlbGxuZXNzIFRvdXJpc20gUHJlZGljdG9yIixwYWdlX2ljb249IuKciO+4jyIpCnN0LnRpdGxlKCJXZWxsbmVzcyBUb3VyaXNtIFBhY2thZ2UgLSBQdXJjaGFzZSBQcmVkaWN0aW9uIikKc3Qud3JpdGUoIkVudGVyIGN1c3RvbWVyIGRldGFpbHMgdG8gZXN0aW1hdGUgcHVyY2hhc2UgbGlrZWxpaG9vZC4iKQpAc3QuY2FjaGVfcmVzb3VyY2UKZGVmIGxvYWRfbW9kZWwoKToKICAgIHJldHVybiBqb2JsaWIubG9hZChQYXRoKF9fZmlsZV9fKS5wYXJlbnQvImRlcGxveW1lbnQiLyJtb2RlbC5qb2JsaWIiKQptb2RlbD1sb2FkX21vZGVsKCkKd2l0aCBzdC5mb3JtKCJjdXN0b21lciIpOgogICAgYSxiPXN0LmNvbHVtbnMoMikKICAgIHdpdGggYToKICAgICAgICBBZ2U9c3QubnVtYmVyX2lucHV0KCJBZ2UiLDE4LDEwMCwzNSkKICAgICAgICBUeXBlb2ZDb250YWN0PXN0LnNlbGVjdGJveCgiVHlwZSBvZiBDb250YWN0IixbIlNlbGYgRW5xdWlyeSIsIkNvbXBhbnkgSW52aXRlZCJdKQogICAgICAgIENpdHlUaWVyPXN0LnNlbGVjdGJveCgiQ2l0eSBUaWVyIixbMSwyLDNdKQogICAgICAgIER1cmF0aW9uT2ZQaXRjaD1zdC5udW1iZXJfaW5wdXQoIkR1cmF0aW9uIG9mIFBpdGNoIixtaW5fdmFsdWU9MC4wLHZhbHVlPTEwLjApCiAgICAgICAgT2NjdXBhdGlvbj1zdC5zZWxlY3Rib3goIk9jY3VwYXRpb24iLFsiU2FsYXJpZWQiLCJTbWFsbCBCdXNpbmVzcyIsIkxhcmdlIEJ1c2luZXNzIiwiRnJlZSBMYW5jZXIiXSkKICAgICAgICBHZW5kZXI9c3Quc2VsZWN0Ym94KCJHZW5kZXIiLFsiTWFsZSIsIkZlbWFsZSIsIkZlIE1hbGUiXSkKICAgICAgICBOdW1iZXJPZlBlcnNvblZpc2l0aW5nPXN0Lm51bWJlcl9pbnB1dCgiTnVtYmVyIG9mIFBlb3BsZSBWaXNpdGluZyIsMSwyMCwyKQogICAgICAgIE51bWJlck9mRm9sbG93dXBzPXN0Lm51bWJlcl9pbnB1dCgiTnVtYmVyIG9mIEZvbGxvdy11cHMiLDAsMjAsMykKICAgICAgICBQcm9kdWN0UGl0Y2hlZD1zdC5zZWxlY3Rib3goIlByb2R1Y3QgUGl0Y2hlZCIsWyJCYXNpYyIsIkRlbHV4ZSIsIlN0YW5kYXJkIiwiU3VwZXIgRGVsdXhlIiwiS2luZyJdKQogICAgd2l0aCBiOgogICAgICAgIFByZWZlcnJlZFByb3BlcnR5U3Rhcj1zdC5zZWxlY3Rib3goIlByZWZlcnJlZCBQcm9wZXJ0eSBTdGFyIixbMyw0LDVdKQogICAgICAgIE1hcml0YWxTdGF0dXM9c3Quc2VsZWN0Ym94KCJNYXJpdGFsIFN0YXR1cyIsWyJTaW5nbGUiLCJNYXJyaWVkIiwiRGl2b3JjZWQiLCJVbm1hcnJpZWQiXSkKICAgICAgICBOdW1iZXJPZlRyaXBzPXN0Lm51bWJlcl9pbnB1dCgiTnVtYmVyIG9mIFRyaXBzIiwwLjAsNTAuMCwzLjApCiAgICAgICAgUGFzc3BvcnQ9c3Quc2VsZWN0Ym94KCJIYXMgUGFzc3BvcnQ/IixbMCwxXSkKICAgICAgICBQaXRjaFNhdGlzZmFjdGlvblNjb3JlPXN0LnNlbGVjdGJveCgiUGl0Y2ggU2F0aXNmYWN0aW9uIFNjb3JlIixbMSwyLDMsNCw1XSkKICAgICAgICBPd25DYXI9c3Quc2VsZWN0Ym94KCJPd25zIGEgQ2FyPyIsWzAsMV0pCiAgICAgICAgTnVtYmVyT2ZDaGlsZHJlblZpc2l0aW5nPXN0Lm51bWJlcl9pbnB1dCgiTnVtYmVyIG9mIENoaWxkcmVuIFZpc2l0aW5nIiwwLDEwLDApCiAgICAgICAgRGVzaWduYXRpb249c3Quc2VsZWN0Ym94KCJEZXNpZ25hdGlvbiIsWyJFeGVjdXRpdmUiLCJNYW5hZ2VyIiwiU2VuaW9yIE1hbmFnZXIiLCJBVlAiLCJWUCJdKQogICAgICAgIE1vbnRobHlJbmNvbWU9c3QubnVtYmVyX2lucHV0KCJNb250aGx5IEluY29tZSIsbWluX3ZhbHVlPTAuMCx2YWx1ZT0yMDAwMC4wKQogICAgc3VibWl0dGVkPXN0LmZvcm1fc3VibWl0X2J1dHRvbigiUHJlZGljdCIpCmlmIHN1Ym1pdHRlZDoKICAgIHJvdz1wZC5EYXRhRnJhbWUoW2xvY2FscygpW2tdIGZvciBrIGluIFtdXSkKICAgIHJvdz1wZC5EYXRhRnJhbWUoW3siQWdlIjpBZ2UsIlR5cGVvZkNvbnRhY3QiOlR5cGVvZkNvbnRhY3QsIkNpdHlUaWVyIjpDaXR5VGllciwiRHVyYXRpb25PZlBpdGNoIjpEdXJhdGlvbk9mUGl0Y2gsIk9jY3VwYXRpb24iOk9jY3VwYXRpb24sIkdlbmRlciI6R2VuZGVyLCJOdW1iZXJPZlBlcnNvblZpc2l0aW5nIjpOdW1iZXJPZlBlcnNvblZpc2l0aW5nLCJOdW1iZXJPZkZvbGxvd3VwcyI6TnVtYmVyT2ZGb2xsb3d1cHMsIlByb2R1Y3RQaXRjaGVkIjpQcm9kdWN0UGl0Y2hlZCwiUHJlZmVycmVkUHJvcGVydHlTdGFyIjpQcmVmZXJyZWRQcm9wZXJ0eVN0YXIsIk1hcml0YWxTdGF0dXMiOk1hcml0YWxTdGF0dXMsIk51bWJlck9mVHJpcHMiOk51bWJlck9mVHJpcHMsIlBhc3Nwb3J0IjpQYXNzcG9ydCwiUGl0Y2hTYXRpc2ZhY3Rpb25TY29yZSI6UGl0Y2hTYXRpc2ZhY3Rpb25TY29yZSwiT3duQ2FyIjpPd25DYXIsIk51bWJlck9mQ2hpbGRyZW5WaXNpdGluZyI6TnVtYmVyT2ZDaGlsZHJlblZpc2l0aW5nLCJEZXNpZ25hdGlvbiI6RGVzaWduYXRpb24sIk1vbnRobHlJbmNvbWUiOk1vbnRobHlJbmNvbWV9XSkKICAgIHA9ZmxvYXQobW9kZWwucHJlZGljdF9wcm9iYShyb3cpWzAsMV0pCiAgICBzdC5tZXRyaWMoIkVzdGltYXRlZCBwdXJjaGFzZSBwcm9iYWJpbGl0eSIsZiJ7cDouMSV9IikKICAgIHN0LnN1Y2Nlc3MoIlBvdGVudGlhbCBwdXJjaGFzZXIgKGNsYXNzIDEpIiBpZiBwPj0uNSBlbHNlICJMZXNzIGxpa2VseSB0byBwdXJjaGFzZSAoY2xhc3MgMCkiKQogICAgc3QuY2FwdGlvbigiTW9kZWwgZXN0aW1hdGUgb25seTsgbm90IGEgZ3VhcmFudGVlIG9mIGN1c3RvbWVyIGJlaGF2aW91ci4iKQonJycKICAgIChyb290LydhcHAucHknKS53cml0ZV90ZXh0KGFwcCkKICAgIChyb290LydyZXF1aXJlbWVudHMudHh0Jykud3JpdGVfdGV4dCgncGFuZGFzPj0yLjBcbm51bXB5Pj0xLjI0XG5zY2lraXQtbGVhcm4+PTEuM1xuam9ibGliPj0xLjNcbnN0cmVhbWxpdD49MS4zMFxuJykKICAgIHdvcmtmbG93PScnJ25hbWU6IFRvdXJpc20gUGFja2FnZSBQcmVkaWN0aW9uIENJCm9uOgogIHB1c2g6CiAgICBicmFuY2hlczogW21haW5dCiAgcHVsbF9yZXF1ZXN0OgogICAgYnJhbmNoZXM6IFttYWluXQogIHdvcmtmbG93X2Rpc3BhdGNoOgpwZXJtaXNzaW9uczoKICBjb250ZW50czogd3JpdGUKam9iczoKICB0cmFpbjoKICAgIHJ1bnMtb246IHVidW50dS1sYXRlc3QKICAgIHN0ZXBzOgogICAgICAtIHVzZXM6IGFjdGlvbnMvY2hlY2tvdXRAdjQKICAgICAgLSB1c2VzOiBhY3Rpb25zL3NldHVwLXB5dGhvbkB2NQogICAgICAgIHdpdGg6CiAgICAgICAgICBweXRob24tdmVyc2lvbjogIjMuMTEiCiAgICAgIC0gcnVuOiBwaXAgaW5zdGFsbCAtciByZXF1aXJlbWVudHMudHh0CiAgICAgIC0gcnVuOiBweXRob24gdG91cmlzbV9wYWNrYWdlX21sb3BzLnB5IC0tZGF0YSBkYXRhL3RvdXJpc20uY3N2IC0tcm9vdCAuCiAgICAgIC0gbmFtZTogQ29tbWl0IHRyYWluZWQgbW9kZWwgYW5kIHJlcG9ydHMKICAgICAgICBpZjogZ2l0aHViLmV2ZW50X25hbWUgIT0gJ3B1bGxfcmVxdWVzdCcKICAgICAgICBydW46IHwKICAgICAgICAgIGdpdCBjb25maWcgdXNlci5uYW1lICJnaXRodWItYWN0aW9uc1tib3RdIgogICAgICAgICAgZ2l0IGNvbmZpZyB1c2VyLmVtYWlsICI0MTg5ODI4MitnaXRodWItYWN0aW9uc1tib3RdQHVzZXJzLm5vcmVwbHkuZ2l0aHViLmNvbSIKICAgICAgICAgIGdpdCBhZGQgZGVwbG95bWVudC9tb2RlbC5qb2JsaWIgcmVwb3J0cyBhcnRpZmFjdHMKICAgICAgICAgIGdpdCBkaWZmIC0tY2FjaGVkIC0tcXVpZXQgfHwgZ2l0IGNvbW1pdCAtbSAiQXV0b21hdGVkIG1vZGVsIHRyYWluaW5nIgogICAgICAgICAgZ2l0IHB1c2gKJycnCiAgICAocm9vdC8nLmdpdGh1Yi93b3JrZmxvd3MvcGlwZWxpbmUueW1sJykud3JpdGVfdGV4dCh3b3JrZmxvdykKICAgIChyb290LydSRUFETUUubWQnKS53cml0ZV90ZXh0KCcjIFZpc2l0IHdpdGggVXMgLSBUb3VyaXNtIFBhY2thZ2UgUHJlZGljdGlvblxuXG5SdW46IGBweXRob24gdG91cmlzbV9wYWNrYWdlX21sb3BzLnB5IC0tZGF0YSBkYXRhL3RvdXJpc20uY3N2YFxuXG5UaGUgcGlwZWxpbmUgdmFsaWRhdGVzIGRhdGEsIHByZXBhcmVzIHN0cmF0aWZpZWQgdHJhaW4vdGVzdCBzcGxpdHMsIGNvbXBhcmVzIGJhc2VsaW5lIGNsYXNzaWZpZXJzLCB0dW5lcyBhIFJhbmRvbSBGb3Jlc3QsIGV2YWx1YXRlcyBpdCwgc2F2ZXMgdGhlIGZpdHRlZCBwcmVwcm9jZXNzaW5nL21vZGVsIHBpcGVsaW5lLCBhbmQgY3JlYXRlcyBhIFN0cmVhbWxpdCBhcHAgYW5kIEdpdEh1YiBBY3Rpb25zIHdvcmtmbG93LlxuJykKICAgIHByaW50KCdcblBST0pFQ1QgQ09NUExFVEUuIEZvbGRlcjonLHJvb3QpCiAgICBwcmludCgnTW9kZWw6Jyxyb290LydkZXBsb3ltZW50L21vZGVsLmpvYmxpYicpCiAgICBwcmludCgnQXBwOicscm9vdC8nYXBwLnB5JykKICAgIHByaW50KCdXb3JrZmxvdzonLHJvb3QvJy5naXRodWIvd29ya2Zsb3dzL3BpcGVsaW5lLnltbCcpCiAgICBvYnMoJ1RoZSBydW4gZ2VuZXJhdGVkIHRoZSBtb2RlbCwgbWV0cmljcywgZGF0YSBzcGxpdHMsIFN0cmVhbWxpdCBhcHAsIHJlcXVpcmVtZW50cyBmaWxlIGFuZCBHaXRIdWIgQWN0aW9ucyB3b3JrZmxvdy4gUmV2aWV3IHRoZSBtZXRyaWNzIGFuZCB2YWxpZGF0ZSB0aGUgYXBwIGlucHV0cyBiZWZvcmUgZGVwbG95bWVudC4nKQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiAgICBtYWluKCkK"
script_path = Path('tourism_package_mlops.py')
script_path.write_bytes(base64.b64decode(script_b64))
print('Pipeline script created:', script_path.resolve())
print('Script size (KB):', round(script_path.stat().st_size / 1024, 1))

**Observation:** The notebook creates the same standalone Python file used by GitHub Actions. This keeps the notebook and repository pipeline aligned and avoids manually copying code between environments.

## 3. Data registration and validation
The validation script checks the expected schema, verifies that the target contains both classes, and prints the dataset dimensions and target distribution.

In [ ]:
import pandas as pd
raw_df = pd.read_csv(data_path)
print('Dataset shape:', raw_df.shape)
print('Column names:', raw_df.columns.tolist())
print('Missing values (top 10):\n', raw_df.isna().sum().sort_values(ascending=False).head(10))
print('Target distribution:\n', raw_df['ProdTaken'].value_counts(dropna=False).sort_index())
expected_columns = ['CustomerID','ProdTaken','Age','TypeofContact','CityTier','DurationOfPitch','Occupation','Gender','NumberOfPersonVisiting','NumberOfFollowups','ProductPitched','PreferredPropertyStar','MaritalStatus','NumberOfTrips','Passport','PitchSatisfactionScore','OwnCar','NumberOfChildrenVisiting','Designation','MonthlyIncome']
missing_columns = sorted(set(expected_columns) - set(raw_df.columns))
assert not missing_columns, f'Missing expected columns: {missing_columns}'
assert raw_df['ProdTaken'].dropna().nunique() == 2, 'Target must contain both 0 and 1'
print('Schema validation: PASSED')

**Observation:** The supplied dataset contains **4,128 rows and 21 columns**. The data dictionary fields are present, along with an extra `Unnamed: 0` index-like column. The target distribution is imbalanced: 3,331 non-purchasers and 797 purchasers. Stratified splitting and class-aware evaluation are therefore important.

## 4. Run data preparation, model experimentation, evaluation and artifact generation
The pipeline removes duplicate rows and records without a target, drops identifier/index columns, imputes missing predictors inside the preprocessing pipeline, and uses an 80:20 stratified train/test split. It compares Logistic Regression, Random Forest and Extra Trees, then tunes Random Forest with cross-validation.

In [ ]:
project_root = Path('visit_with_us_tourism_project')
cmd = [sys.executable, str(script_path), '--data', str(data_path.resolve()), '--root', str(project_root.resolve())]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-4000:])
if result.returncode != 0:
    raise RuntimeError(f'Pipeline failed with exit code {result.returncode}')
print('Pipeline completed successfully.')

**Observation:** The pipeline produces training and test CSV files, a baseline model comparison, tuned model parameters, test metrics, a serialized model, a Streamlit app, and a GitHub Actions workflow. `CustomerID` and `Unnamed: 0` are excluded because they are identifiers/index values rather than meaningful customer predictors.

## 5. Review model comparison and final test metrics

In [ ]:
comparison = pd.read_csv(project_root / 'reports' / 'model_comparison.csv')
metrics = json.loads((project_root / 'reports' / 'metrics.json').read_text())
print('Baseline model comparison:')
display(comparison)
print('Tuned model:', metrics['best_model'])
print('Best parameters:', metrics['best_params'])
metric_view = pd.DataFrame([
    {'Metric': 'Accuracy', 'Value': metrics['test_accuracy']},
    {'Metric': 'Precision', 'Value': metrics['test_precision']},
    {'Metric': 'Recall', 'Value': metrics['test_recall']},
    {'Metric': 'F1-score', 'Value': metrics['test_f1']},
    {'Metric': 'ROC-AUC', 'Value': metrics['test_roc_auc']},
])
display(metric_view)
print('Confusion matrix [[TN, FP], [FN, TP]]:')
display(pd.DataFrame(metrics['confusion_matrix'], index=['Actual 0','Actual 1'], columns=['Predicted 0','Predicted 1']))

**Observation and interpretation:** On this fixed 20% holdout, the tuned Random Forest achieved approximately **90.7% accuracy**, **94.6% precision**, **54.7% recall**, **69.3% F1**, and **96.9% ROC-AUC**. Precision is high, meaning a large share of customers flagged by the model are purchasers in this test set. Recall is lower, so the model misses some actual purchasers. The confusion matrix should be discussed in business terms: false positives may create unnecessary marketing contacts, while false negatives represent missed sales opportunities. Do not rely on accuracy alone because the target classes are imbalanced.

## 6. Model deployment — Streamlit frontend
The generated `app.py` loads the saved preprocessing/model pipeline and collects customer attributes into a dataframe before displaying the estimated purchase probability. The fitted pipeline includes imputation and one-hot encoding, so the app uses the same preprocessing as training.

In [ ]:
print((project_root / 'app.py').read_text()[:5000])
print('\nRequirements:')
print((project_root / 'requirements.txt').read_text())
print('\nModel artifact exists:', (project_root / 'deployment' / 'model.joblib').exists())

**Observation:** The app is ready to run locally with `streamlit run app.py`. The model file and app must be committed to the public repository before deployment. The supplied rubric explicitly names Streamlit Community Cloud; the later submission note also asks for a Hugging Face Spaces URL. These are different hosting platforms, so publish to the platform your programme manager confirms, or deploy to both and include both real public URLs. The project ZIP includes a Dockerfile as a starting point for a Hugging Face Docker Space.

## 7. GitHub Actions CI/CD workflow
The workflow runs on pushes and pull requests to `main`, installs dependencies, executes the ML pipeline, uploads generated artifacts, and commits refreshed model/report artifacts on successful pushes to `main`. The commit uses `[skip ci]` to prevent an endless workflow-trigger loop.

In [ ]:
print((project_root / '.github' / 'workflows' / 'pipeline.yml').read_text())
print('\nProject files:')
for p in sorted(project_root.rglob('*')):
    if p.is_file() and '__pycache__' not in str(p):
        print(p.relative_to(project_root))

**Observation:** After pushing the project folder to a GitHub repository, open **Actions** and confirm that the workflow completes successfully. The workflow artifact contains the train/test splits and reports. The current workflow is a single CI job that runs the end-to-end script and uploads its generated outputs; if your assessor strictly requires a separate preparation job and a downstream training job, split the script into two CLI stages and pass the splits using `upload-artifact` / `download-artifact` between jobs.

## 8. Business insights, limitations and recommendations

- Use the predicted probability to rank customers for a campaign rather than treating the class label as an absolute decision.
- Choose a decision threshold based on campaign capacity and the relative cost of missed purchasers versus unnecessary outreach.
- Investigate recall: the current test result suggests that some actual purchasers are not identified at the default 0.50 threshold.
- Monitor precision, recall, F1 and ROC-AUC on future campaign outcomes; retrain when customer behaviour or performance changes.
- Keep customer information secure and restrict access to the repository and deployed application as appropriate.

**Important timing limitation:** The supplied dataset contains `DurationOfPitch`, `NumberOfFollowups`, and `ProductPitched`, which are interaction/sales-pitch attributes. The business objective says to predict purchase *before contacting* a customer. These fields may not exist at that decision point and could cause information leakage. The current exercise follows the supplied dataset/rubric; for genuine pre-contact targeting, remove unavailable post-contact fields and retrain/evaluate a separate model.

## 9. Submission checklist and links to add

Before submitting, complete these steps:

1. Create a public GitHub repository and upload the contents of `visit_with_us_tourism_project/` (including `data/tourism.csv`, workflow, app and model).
2. Add the GitHub repository URL below.
3. Deploy `app.py` on Streamlit Community Cloud as required by the rubric. If Hugging Face Spaces is the confirmed requirement, create a public Docker Space and use the included `Dockerfile`.
4. Replace the placeholders with actual public deployment links. Never submit placeholder URLs.
5. Capture screenshots of the repository folder structure, successful GitHub Actions workflow, and deployed frontend.
6. Run all notebook cells sequentially and export the executed notebook as HTML.

**GitHub repository:** `https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPOSITORY>`  
**Streamlit app (rubric):** `<PASTE_PUBLIC_STREAMLIT_URL>`  
**Hugging Face Space (if required):** `https://huggingface.co/spaces/<YOUR_USERNAME>/<YOUR_SPACE>`

**Final observation:** The local pipeline and deployment assets are generated, but public deployment is not complete until you create the repository/hosting space, push the files, verify the live app, and replace the placeholders with working URLs.